# MAP priors and reparameterized rates

This tutorial shows two ways to add prior information to the HOGENOM likelihood:

1. Optimize a MAP objective directly in the model's log2-rate parameter `theta`.
2. Optimize an unconstrained tensor `alpha` with a differentiable map `theta = f(alpha)`.

The likelihood returned by `GeneReconModel` is a negative log-likelihood in bits, so prior penalties below are also expressed in bits.

## Setup

In [ ]:
from pathlib import Path
import math
import os
import sys
import time

import numpy as np
import torch

CWD = Path.cwd()
REPO = CWD if (CWD / "gpurec").exists() else CWD.parent if (CWD.parent / "gpurec").exists() else CWD
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from gpurec import GeneReconModel, SolverOptions, clamp_log_rate_, project_rate_gradient_

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("The current HOGENOM likelihood path requires CUDA/Triton.")
DEVICE

In [ ]:
def find_hogenom_root() -> Path:
    env_root = os.environ.get("HOGENOM_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        REPO / "tests/data",
        REPO / "tests/data/HOGENOM/hogenom",
        REPO / "tests/data/HOGENOM",
        REPO / "tests/data/hogenom",
        REPO / "data/HOGENOM/hogenom",
    ])
    for candidate in candidates:
        if (candidate / "hogenom_S.tree").exists() and (candidate / "hogenom_trees").is_dir():
            return candidate
    raise FileNotFoundError("Set HOGENOM_ROOT to a directory containing hogenom_S.tree and hogenom_trees/.")


HOGENOM_ROOT = find_hogenom_root()
SPECIES_TREE = HOGENOM_ROOT / "hogenom_S.tree"
GENE_TREE_DIR = HOGENOM_ROOT / "hogenom_trees"

MODE = os.environ.get("GPUREC_MODE", "specieswise")
MAX_FAMILIES = 20
MIN_RATE = 1e-10
MAX_RATE = 2.0

gene_trees = sorted(GENE_TREE_DIR.glob("*.trees"))[:MAX_FAMILIES]
{
    "mode": MODE,
    "families": len(gene_trees),
    "species_tree": str(SPECIES_TREE),
    "gene_tree_dir": str(GENE_TREE_DIR),
}

In [ ]:
solver_options = SolverOptions(
    e_init=-1000,
    e_max_iter=2000,
    e_tol=1e-8,
    pi_iters=16,
    neumann_terms=16,
    bicgstab_max_iter=500,
    bicgstab_tol=1e-7,
    bicgstab_breakdown_tol=1e-30,
    adjoint_pruning_threshold=1e-6,
    use_adjoint_pruning=True,
    pibar_side_threshold=0.0,
)

model = GeneReconModel(
    SPECIES_TREE,
    gene_trees,
    mode=MODE,
    device=DEVICE,
    family_chunk_size=MAX_FAMILIES,
    clade_budget=315_000,
    batch_packing="depth_first_fit",
    max_wave_size=8192,
    solver_options=solver_options,
)
clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)

{
    "theta_shape": tuple(model.theta.shape),
    "batches": [len(batch) for batch in model.family_batches],
}

## A prior on theta

`theta` stores log2 event-rate logits. A Gaussian prior on `theta` is a simple regularizer in that same coordinate system. Since the likelihood is in bits, divide the usual natural-log Gaussian penalty by `log(2)`.

In [ ]:
def log2_bounds(min_rate, max_rate):
    lower = math.log2(float(min_rate))
    upper = None if max_rate is None else math.log2(float(max_rate))
    return lower, upper


def gaussian_theta_neg_log_prior_bits(theta, *, mean_log2_rate, sd_log2_rate):
    mean = torch.as_tensor(mean_log2_rate, dtype=theta.dtype, device=theta.device)
    sd = torch.as_tensor(sd_log2_rate, dtype=theta.dtype, device=theta.device)
    z = (theta - mean) / sd
    return 0.5 * (z * z).sum() / math.log(2.0)


def event_probabilities(theta):
    zeros = theta.new_zeros((*theta.shape[:-1], 1))
    logits = torch.cat((zeros, theta), dim=-1)
    return torch.softmax(logits * math.log(2.0), dim=-1)


PRIOR_MEAN_LOG2_RATE = math.log2(1e-5)
PRIOR_SD_LOG2_RATE = 2.0

In [ ]:
def theta_map_loss(model):
    nll_bits = model()
    prior_bits = gaussian_theta_neg_log_prior_bits(
        model.theta,
        mean_log2_rate=PRIOR_MEAN_LOG2_RATE,
        sd_log2_rate=PRIOR_SD_LOG2_RATE,
    )
    return nll_bits + prior_bits, nll_bits, prior_bits


model.zero_grad(set_to_none=True)
loss_bits, nll_bits, prior_bits = theta_map_loss(model)
loss_bits.backward()

raw_grad_norm = float(model.theta.grad.detach().norm().cpu())
projected_grad = model.theta.grad.detach().clone()
project_rate_gradient_(model.theta, projected_grad, min_rate=MIN_RATE, max_rate=MAX_RATE)

{
    "map_objective_bits": float(loss_bits.detach().cpu()),
    "nll_bits": float(nll_bits.detach().cpu()),
    "prior_bits": float(prior_bits.detach().cpu()),
    "raw_grad_norm": raw_grad_norm,
    "projected_grad_norm": float(projected_grad.norm().detach().cpu()),
}

## Optimizing MAP directly in theta

For PyTorch optimizers, backpropagate the total MAP objective. For SciPy L-BFGS-B, return the same scalar objective and its raw gradient; L-BFGS-B applies the bound projection internally.

In [ ]:
RUN_TORCH_STEPS = False

if RUN_TORCH_STEPS:
    optimizer = torch.optim.Adam([model.theta], lr=0.25)
    history = []
    for step in range(1, 21):
        optimizer.zero_grad(set_to_none=True)
        loss_bits, nll_bits, prior_bits = theta_map_loss(model)
        loss_bits.backward()
        project_rate_gradient_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
        optimizer.step()
        clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
        history.append({
            "step": step,
            "map_objective_bits": float(loss_bits.detach().cpu()),
            "nll_bits": float(nll_bits.detach().cpu()),
            "prior_bits": float(prior_bits.detach().cpu()),
        })
    history[-5:]
else:
    "Set RUN_TORCH_STEPS=True to run a short MAP optimization in theta."

In [ ]:
def flat_theta_numpy(model):
    return model.theta.detach().cpu().double().numpy().reshape(-1).copy()


@torch.no_grad()
def load_flat_theta_(model, flat_theta):
    theta_cpu = torch.from_numpy(np.asarray(flat_theta, dtype=np.float64).reshape(tuple(model.theta.shape)))
    model.theta.copy_(theta_cpu.to(device=model.theta.device, dtype=model.theta.dtype))
    clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)


def lbfgsb_bounds(theta):
    lower, upper = log2_bounds(MIN_RATE, MAX_RATE)
    return [(lower, upper)] * int(theta.numel())


def scipy_theta_map_objective(flat_theta):
    load_flat_theta_(model, flat_theta)
    model.zero_grad(set_to_none=True)
    loss_bits, nll_bits, prior_bits = theta_map_loss(model)
    loss_bits.backward()
    grad = model.theta.grad.detach().cpu().double().numpy().reshape(-1).copy()
    return float(loss_bits.detach().cpu()), grad


RUN_SCIPY_THETA = False

if RUN_SCIPY_THETA:
    from scipy.optimize import minimize

    result = minimize(
        scipy_theta_map_objective,
        flat_theta_numpy(model),
        method="L-BFGS-B",
        jac=True,
        bounds=lbfgsb_bounds(model.theta),
        options={"maxiter": 25, "maxls": 50, "gtol": 1.0, "ftol": 1e-6},
    )
    load_flat_theta_(model, result.x)
    result
else:
    "Set RUN_SCIPY_THETA=True to run bounded L-BFGS-B on the MAP objective in theta."

## Differentiating through theta = f(alpha)

Use `model(theta)` with an external tensor to keep the computation graph intact. Do not copy `f(alpha)` into `model.theta` if you want gradients with respect to `alpha`; the copy is not a differentiable operation from the optimizer's point of view.

In [ ]:
def bounded_sigmoid_theta(alpha, *, min_rate=MIN_RATE, max_rate=MAX_RATE):
    lower, upper = log2_bounds(min_rate, max_rate)
    if upper is None:
        raise ValueError("bounded_sigmoid_theta requires a finite max_rate")
    return lower + (upper - lower) * torch.sigmoid(alpha)


def alpha_map_loss(alpha):
    theta = bounded_sigmoid_theta(alpha)
    nll_bits = model(theta)
    prior_bits = gaussian_theta_neg_log_prior_bits(
        theta,
        mean_log2_rate=PRIOR_MEAN_LOG2_RATE,
        sd_log2_rate=PRIOR_SD_LOG2_RATE,
    )
    return nll_bits + prior_bits, nll_bits, prior_bits, theta


alpha = torch.nn.Parameter(torch.zeros_like(model.theta))
loss_bits, nll_bits, prior_bits, theta = alpha_map_loss(alpha)
loss_bits.backward()

{
    "map_objective_bits": float(loss_bits.detach().cpu()),
    "nll_bits": float(nll_bits.detach().cpu()),
    "prior_bits": float(prior_bits.detach().cpu()),
    "alpha_grad_norm": float(alpha.grad.detach().norm().cpu()),
    "theta_min": float(theta.detach().min().cpu()),
    "theta_max": float(theta.detach().max().cpu()),
}

The same `alpha` objective can be optimized with either PyTorch or SciPy. Because `alpha` is unconstrained, SciPy does not need bounds in this example. If the optimum is expected to lie exactly on a rate bound, direct L-BFGS-B on `theta` may be better than a sigmoid map because the sigmoid derivative gets small near the bounds.

In [ ]:
RUN_ALPHA_TORCH_STEPS = False

if RUN_ALPHA_TORCH_STEPS:
    alpha = torch.nn.Parameter(torch.zeros_like(model.theta))
    optimizer = torch.optim.Adam([alpha], lr=0.05)
    history = []
    for step in range(1, 21):
        optimizer.zero_grad(set_to_none=True)
        loss_bits, nll_bits, prior_bits, theta = alpha_map_loss(alpha)
        loss_bits.backward()
        optimizer.step()
        history.append({
            "step": step,
            "map_objective_bits": float(loss_bits.detach().cpu()),
            "nll_bits": float(nll_bits.detach().cpu()),
            "prior_bits": float(prior_bits.detach().cpu()),
            "theta_min": float(theta.detach().min().cpu()),
            "theta_max": float(theta.detach().max().cpu()),
        })
    history[-5:]
else:
    "Set RUN_ALPHA_TORCH_STEPS=True to optimize alpha with PyTorch."

In [ ]:
def scipy_alpha_map_objective(flat_alpha):
    alpha = torch.tensor(
        np.asarray(flat_alpha, dtype=np.float64).reshape(tuple(model.theta.shape)),
        dtype=model.theta.dtype,
        device=model.theta.device,
        requires_grad=True,
    )
    loss_bits, nll_bits, prior_bits, theta = alpha_map_loss(alpha)
    loss_bits.backward()
    grad = alpha.grad.detach().cpu().double().numpy().reshape(-1).copy()
    return float(loss_bits.detach().cpu()), grad


RUN_SCIPY_ALPHA = False

if RUN_SCIPY_ALPHA:
    from scipy.optimize import minimize

    alpha0 = np.zeros(int(model.theta.numel()), dtype=np.float64)
    result = minimize(
        scipy_alpha_map_objective,
        alpha0,
        method="L-BFGS-B",
        jac=True,
        options={"maxiter": 25, "maxls": 50, "gtol": 1.0, "ftol": 1e-6},
    )
    theta_hat = bounded_sigmoid_theta(
        torch.as_tensor(result.x.reshape(tuple(model.theta.shape)), dtype=model.theta.dtype, device=model.theta.device)
    )
    {"result": result, "theta_min": float(theta_hat.min().cpu()), "theta_max": float(theta_hat.max().cpu())}
else:
    "Set RUN_SCIPY_ALPHA=True to optimize unconstrained alpha with SciPy."

## Takeaways

- MAP means minimizing `negative_log_likelihood + negative_log_prior`.
- Keep the prior penalty in bits to match the model objective.
- Return the gradient of the same total objective to L-BFGS-B.
- Use `model(theta)` for differentiable external parameters; copying values into `model.theta` is fine for direct `theta` optimization but not for differentiating through `theta = f(alpha)`.
- Bound-constrained optimization in `theta` and unconstrained optimization in `alpha` are different numerical problems; sigmoid maps are convenient but can slow near hard bounds.